# Safe Walk Home: BFS Pathfinding Algorithm

## Problem Description

This notebook implements a **Breadth-First Search (BFS)** algorithm to determine if a player can safely navigate through a dangerous grid from the top-left corner to the bottom-right corner while maintaining their health.

### Game Rules:
- **Starting Position**: Top-left corner (0, 0)
- **Target Position**: Bottom-right corner (last row, last column)
- **Health System**: Each cell contains a damage value that reduces the player's health when entered
- **Survival Condition**: Player must maintain at least 1 health point to survive
- **Movement**: Player can move UP, DOWN, LEFT, or RIGHT (no diagonal movement)

### Grid Structure Visualization:

For a 3×4 grid:
```
    Col: 0  1  2  3
Row 0: [a, b, c, d]  ← START is at (0,0) = 'a'
Row 1: [e, f, g, h]
Row 2: [i, j, k, l]  ← TARGET is at (2,3) = 'l'
```

The algorithm uses BFS to explore all possible paths and determines if any path allows the player to reach the target with sufficient health.

## Import Required Libraries

We need to import the `deque` from the `collections` module to implement an efficient queue for our BFS algorithm. A deque (double-ended queue) provides O(1) append and popleft operations, which are essential for BFS performance.

In [2]:
from collections import deque

## BFS Algorithm Implementation

The core of our solution is the `can_reach_end_bfs` function that implements a breadth-first search algorithm. Let's break down the key components:

### Algorithm Components:

1. **Grid Dimensions**: We extract the number of rows and columns from the input grid
2. **Health Tracking Matrix**: A 2D array that tracks the maximum health achieved when visiting each cell
3. **BFS Queue**: Stores positions to explore along with the current health at each position
4. **Movement Directions**: Defines the four possible movements (UP, DOWN, LEFT, RIGHT)
5. **Boundary Checking**: Ensures we don't move outside the grid boundaries
6. **Health Management**: Calculates health after taking damage and ensures survival

### Key Optimizations:

- **Pruning**: We only explore a path if it offers better health than previously recorded for that cell
- **Early Termination**: We return `True` immediately upon reaching the target
- **Efficient Queue Operations**: Using deque for O(1) queue operations

### Function Definition and Grid Setup

First, let's define our function and set up the basic grid structure:

In [3]:
def can_reach_end_bfs(grid, initial_health):
    """
    Determines if a player can reach the bottom-right cell of a grid using BFS.

    Args:
        grid: 2D list representing the game board where each cell contains damage value
        initial_health: Starting health points of the player

    Returns:
        bool: True if the player can reach the end, False otherwise
    """
    # Get grid dimensions
    # num_rows = number of horizontal rows (height of the matrix)
    # num_cols = number of vertical columns (width of the matrix)
    num_rows, num_cols = len(grid), len(grid[0])
    
    # Track the maximum health recorded when visiting each cell
    # Initialize with -1 to indicate unvisited cells
    # This matrix has the SAME SHAPE as the input grid:
    # max_health_at_cell[row][col] stores the best health achieved at grid[row][col]
    max_health_at_cell = [[-1] * num_cols for _ in range(num_rows)]

### Initialize BFS Queue and Starting Position

In [4]:
    # BFS queue to store (row, col, current_health) tuples
    search_queue = deque()

    # Calculate health after taking damage from starting cell (top-left corner)
    # Starting position is always grid[0][0] (first row, first column)
    health_after_start = initial_health - grid[0][0]
    if health_after_start < 1:
        return False  # Player dies at the starting position

    # Add starting position to queue and mark as visited
    # Queue stores tuples: (row_index, column_index, current_health)
    search_queue.append((0, 0, health_after_start))
    max_health_at_cell[0][0] = health_after_start

NameError: name 'initial_health' is not defined

### Define Movement Directions

The player can move in four directions from any position. Each movement is represented as a change in row and column coordinates:

In [ ]:
    # Define movement directions in (row_change, column_change) format:
    # (-1, 0) = UP    (decrease row by 1, same column)
    # (1, 0)  = DOWN  (increase row by 1, same column)  
    # (0, -1) = LEFT  (same row, decrease column by 1)
    # (0, 1)  = RIGHT (same row, increase column by 1)
    #
    # Visual representation of movements from position (r,c):
    #           (r-1,c) ↑ UP
    # (r,c-1) ← (r,c) → (r,c+1)
    #           (r+1,c) ↓ DOWN
    movement_directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

### Main BFS Loop

The core of the algorithm explores all reachable positions level by level, ensuring we find the optimal path:

In [ ]:
    # Perform BFS traversal
    while search_queue:
        current_row, current_col, current_health = search_queue.popleft()

        # Check if we've reached the target (bottom-right corner)
        # Target position is always at (num_rows-1, num_cols-1)
        # This represents the last row and last column of the matrix
        if current_row == num_rows - 1 and current_col == num_cols - 1:
            return True  # Successfully reached the destination

        # Explore all four possible directions from current position
        for row_delta, col_delta in movement_directions:
            # Calculate the next position by applying the movement
            next_row, next_col = current_row + row_delta, current_col + col_delta

            # Check if the next position is within grid boundaries
            # Valid row range: [0, num_rows-1]
            # Valid col range: [0, num_cols-1]
            if 0 <= next_row < num_rows and 0 <= next_col < num_cols:
                # Calculate health after taking damage from the next cell
                health_after_damage = current_health - grid[next_row][next_col]

                # Only proceed if player survives and this path offers better health
                if health_after_damage >= 1 and health_after_damage > max_health_at_cell[next_row][next_col]:
                    # Update the maximum health recorded for this cell
                    max_health_at_cell[next_row][next_col] = health_after_damage
                    # Add this position to the queue for further exploration
                    search_queue.append((next_row, next_col, health_after_damage))

    # If we've exhausted all possibilities without reaching the target
    return False

## Test Case 1: Simple Path with Obstacles

Let's test our algorithm with a simple 3×5 grid that contains some obstacles (high damage cells) but has a clear path to the destination.

### Grid Visualization (3 rows × 5 columns):
```
    Col: 0  1  2  3  4
Row 0: [0, 1, 0, 0, 0]  ← START at (0,0) = 0 damage
Row 1: [0, 1, 0, 1, 0]
Row 2: [0, 0, 0, 1, 0]  ← TARGET at (2,4) = 0 damage
```

**Starting Health**: 1 point
**Expected Result**: `True` - The player should be able to find a path

**Possible Path**: (0,0) → (1,0) → (2,0) → (2,1) → (2,2) → (2,4)
- Each move costs 0 damage except when hitting the 1-damage cells
- The player can avoid most high-damage cells

In [ ]:
print("Test Case 1 - Simple path with obstacles:")
result1 = can_reach_end_bfs([[0, 1, 0, 0, 0], [0, 1, 0, 1, 0], [0, 0, 0, 1, 0]], 1)
print(f"Result: {result1}")
print(f"Expected: True")
print(f"✅ Test {'PASSED' if result1 == True else 'FAILED'}")

## Test Case 2: Challenging Path with Higher Damage

Now let's test with a more challenging 4×6 grid that has more obstacles and requires careful navigation.

### Grid Visualization (4 rows × 6 columns):
```
    Col: 0  1  2  3  4  5
Row 0: [0, 1, 1, 0, 0, 0]  ← START at (0,0) = 0 damage
Row 1: [1, 0, 1, 0, 0, 0]  
Row 2: [0, 1, 1, 1, 0, 1]
Row 3: [0, 0, 1, 0, 1, 0]  ← TARGET at (3,5) = 0 damage
```

**Starting Health**: 3 points
**Expected Result**: `False` - The path should be impossible or too costly

**Analysis**: This grid has many 1-damage cells strategically placed to block most efficient paths. Even with 3 starting health points, the player may not be able to reach the destination while maintaining at least 1 health point.

In [ ]:
print("Test Case 2 - Challenging path with higher damage:")
result2 = can_reach_end_bfs([[0, 1, 1, 0, 0, 0], [1, 0, 1, 0, 0, 0], 
                             [0, 1, 1, 1, 0, 1], [0, 0, 1, 0, 1, 0]], 3)
print(f"Result: {result2}")
print(f"Expected: False")
print(f"✅ Test {'PASSED' if result2 == False else 'FAILED'}")

## Test Case 3: High Damage Grid with Sufficient Health

Finally, let's test with a small 3×3 grid where every cell deals damage, but the player starts with sufficient health to overcome the obstacles.

### Grid Visualization (3 rows × 3 columns):
```
    Col: 0  1  2
Row 0: [1, 1, 1]  ← START at (0,0) = 1 damage
Row 1: [1, 0, 1]
Row 2: [1, 1, 1]  ← TARGET at (2,2) = 1 damage
```

**Starting Health**: 5 points
**Expected Result**: `True` - High initial health should allow success

**Analysis**: Every cell except (1,1) deals 1 damage. With 5 starting health points, the player should be able to find a path that costs at most 4 health points total, leaving them with at least 1 health point at the destination.

**Optimal Path**: (0,0) → (1,0) → (1,1) → (1,2) → (2,2)
- Cost: 1 + 1 + 0 + 1 + 1 = 4 damage
- Final health: 5 - 4 = 1 (survives!)

In [ ]:
print("Test Case 3 - High damage grid with sufficient health:")
result3 = can_reach_end_bfs([[1, 1, 1], [1, 0, 1], [1, 1, 1]], 5)
print(f"Result: {result3}")
print(f"Expected: True")
print(f"✅ Test {'PASSED' if result3 == True else 'FAILED'}")

## Conclusion and Algorithm Analysis

### Algorithm Summary

Our BFS implementation successfully solves the "Safe Walk Home" problem by:

1. **Systematic Exploration**: Using BFS ensures we explore all possible paths level by level
2. **Optimal Health Tracking**: We only pursue paths that offer better health than previously recorded
3. **Early Termination**: The algorithm stops as soon as any valid path to the destination is found
4. **Efficient Pruning**: By tracking maximum health per cell, we avoid redundant explorations

### Time and Space Complexity

- **Time Complexity**: O(V × H) where V is the number of cells and H is the maximum possible health
- **Space Complexity**: O(V) for the visited matrix and BFS queue

### Key Insights

- The algorithm works well for grids where there are viable paths with sufficient health
- Higher starting health dramatically increases the chances of finding a successful path
- Strategic placement of high-damage cells can effectively block paths even with generous health
- BFS guarantees that if a solution exists, it will be found

This pathfinding approach can be adapted for many similar problems involving resource management during navigation!